In [ ]:
import os
import sys
import gradio as gr


curr_path = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
root_path = curr_path
while not os.path.exists(os.path.join(root_path, "configs")) and len(root_path) > 3:
    root_path = os.path.dirname(root_path)
if root_path not in sys.path: sys.path.append(root_path)

from src.gradio_adapter.avg_attack_adapter import AVGAttackGradioAdapter
backend = AVGAttackGradioAdapter()

def list_cfgs(subdir):
    p = os.path.join(root_path, "configs", subdir)
    return [f for f in sorted(os.listdir(p)) if f.endswith('.yaml')] if os.path.exists(p) else []

CFG_LISTS = {"wm": list_cfgs("methods/watermarks"), "model": list_cfgs("models"), "ds": list_cfgs("datasets")}

def run_gen_ui(wm, ds, model, seed, n):
    log = ""
    for msg in backend.run_triple_generation(wm, ds, model, seed, n):
        log += str(msg); yield log

def run_atk_ui(wm, model, mode, box, ds, next_n, ntst_n):
    log = ""
    exp_dir = backend.update_dir_by_wm(wm, ds)
    for res in backend.run_avg_attack_test(wm, model, mode.lower(), box, exp_dir, next_n, ntst_n):
        msg, df = res if isinstance(res, (tuple, list)) else (res, None)
        if msg: log += str(msg)
        yield log, df

with gr.Blocks(title="AVG Attack Workstation") as app:
    gr.HTML("<h1 style='text-align: center;'>🛡️ AVG Attack Automated Evaluation Workstation</h1>")
    
    with gr.Tabs():
        
        with gr.Tab("Step 1: Batch Triplet Preparation"):
            with gr.Row():
                with gr.Column(scale=1):
                    t1_wm = gr.Dropdown(CFG_LISTS['wm'], label="Watermark Algorithm", value=CFG_LISTS['wm'][0] if CFG_LISTS['wm'] else None)
                    t1_ds = gr.Dropdown(CFG_LISTS['ds'], label="Dataset", value="coco.yaml")
                    t1_model = gr.Dropdown(CFG_LISTS['model'], label="Model", value="sd_v1_5.yaml")
                    t1_seed = gr.Number(42, label="Start Seed", precision=0)
                    t1_n = gr.Number(10, label="Generation Count (N)", precision=0)
                    btn_gen = gr.Button("🚀 Start Batch Generation", variant="primary")
                t1_log = gr.TextArea(label="Logs", lines=18)

        
        with gr.Tab("Step 2: Attack Effectiveness Evaluation"):
            with gr.Row():
                with gr.Column(scale=1):
                    t2_wm = gr.Dropdown(CFG_LISTS['wm'], label="Target Algorithm")
                    t2_ds = gr.Dropdown(CFG_LISTS['ds'], label="Corresponding Dataset", value="coco.yaml")
                    
                    with gr.Row():
                        t2_mode = gr.Radio(["Detect", "Forgery", "Removal"], label="Test Mode", value="Forgery")
                        t2_box = gr.Radio(["Black-box", "Gray-box"], label="Scenario", value="Gray-box")
                    
                    with gr.Row():
                        t2_next = gr.Number(10, label="Extraction Samples (N-Ext)", precision=0)
                        t2_ntst = gr.Number(10, label="Test Samples (N-Test)", precision=0)
                    
                    btn_atk = gr.Button("🔥 Run Evaluation", variant="primary")
                    t2_res = gr.Dataframe(label="Statistical Metrics")
                t2_log = gr.TextArea(label="Console Output", lines=18)

    btn_gen.click(run_gen_ui, [t1_wm, t1_ds, t1_model, t1_seed, t1_n], [t1_log])
    btn_atk.click(run_atk_ui, [t2_wm, gr.State("sd_v1_5.yaml"), t2_mode, t2_box, t2_ds, t2_next, t2_ntst], [t2_log, t2_res])

if __name__ == "__main__":
    app.queue().launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


   [Model] Loading Diffusers: data/models/stable-diffusion-v1-5 (torch.float16)...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


    -> [COCO] Loaded 5000 annotations.
    -> [COCO] Loaded 5000 image infos.
    -> [COCO] Loaded 5000 annotations.
    -> [COCO] Loaded 5000 image infos.
